In [5]:
import os
import numpy as np
import pandas as pd
import rasterio
from rasterio.mask import mask
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject
import geopandas as gpd

# =========================
# Config
# =========================
TIF_T0 = r"../Result/Land_Cover_Each_Year/ESRI_2018_reclass_300m.tif"
TIF_T1 = r"../Result/Land_Cover_Each_Year/ESRI_2019_reclass_300m.tif"

OUT_TIF = r"../Result/Land_Cover_Degradation/status_2018_2019_deg_stable_imp.tif"
OUT_CSV = r"../Result/Land_Cover_Degradation/status_2018_2019_area_km2_stats.csv"

AOI_SHP = r"../../data_hmq/Map/abu_dhabi_all.shp"  # polygon AOI

# NODATA must not collide with 0/1/2 codes => use 255 for uint8
NODATA_VAL = 255

PERIOD = 'Reporting'  # 'Baseline' or 'Reporting'

# Status codes
STABLE = 0
IMPROVEMENT = 1
DEGRADATION = 2

STATUS_LABELS = {
    NODATA_VAL: "NoData",
    STABLE: "Stable",
    IMPROVEMENT: "Improvement",
    DEGRADATION: "Degradation",
}

STATUS_COLORMAP = {
    NODATA_VAL: (0, 0, 0, 0),          # transparent
    STABLE:     (230, 230, 230, 255),  # grey
    IMPROVEMENT:(0, 160, 80, 255),     # green
    DEGRADATION:(200, 60, 60, 255),    # red
}

# Classes used in the matrix (from your figure):
# 1 Forest, 2 Grassland, 3 Cropland, 4 Wetlands, 5 Artificial, 6 Other
# Your reclass also includes 7 Water (not in the matrix figure).
IGNORE_CLASSES = (7,)  # treat any transition involving these as NoData


# =========================
# Helpers
# =========================
def load_aoi_geoms(aoi_shp: str, target_crs):
    gdf = gpd.read_file(aoi_shp)
    if gdf.empty:
        raise ValueError(f"AOI is empty: {aoi_shp}")
    if gdf.crs is None:
        raise ValueError("AOI shapefile has no CRS. Please define CRS first.")
    if target_crs is not None and gdf.crs != target_crs:
        gdf = gdf.to_crs(target_crs)
    geoms = [
        geom.__geo_interface__
        for geom in gdf.geometry
        if geom is not None and (not geom.is_empty)
    ]
    if not geoms:
        raise ValueError("AOI has no valid geometries.")
    return geoms


def ensure_same_grid(ds_a, arr_a, ds_b, arr_b, nodata=NODATA_VAL):
    """
    If CRS/transform/shape differ, reproject B -> A using nearest neighbor.
    Returns arr_a, arr_b_aligned, ds_a
    """
    same = (
        ds_a.crs == ds_b.crs and
        ds_a.transform == ds_b.transform and
        ds_a.width == ds_b.width and
        ds_a.height == ds_b.height
    )
    if same:
        return arr_a, arr_b, ds_a

    dst = np.empty((ds_a.height, ds_a.width), dtype=arr_b.dtype)
    reproject(
        source=arr_b,
        destination=dst,
        src_transform=ds_b.transform,
        src_crs=ds_b.crs,
        dst_transform=ds_a.transform,
        dst_crs=ds_a.crs,
        resampling=Resampling.nearest,
        src_nodata=nodata,
        dst_nodata=nodata,
    )
    return arr_a, dst, ds_a


def build_transition_lut(nodata=NODATA_VAL, period='Baseline'):
    """
    Build LUT for transitions based on your screenshot matrix.
    lut[orig_class, final_class] -> status_code (0/1/2)
    Class codes:
      1 Forest
      2 Grassland
      3 Cropland
      4 Wetlands
      5 Artificial
      6 Other
    Any unspecified entry defaults to DEGRADATION (per your matrix with many '-').
    """
    lut = np.full((8, 8), DEGRADATION, dtype=np.uint8)  # indices 0..7, 0 unused

    # Diagonal stable for 1..6
    for k in range(1, 7):
        lut[k, k] = STABLE

    if period == 'Baseline':
        # Row Forest (1):

        # Row Grassland (2):
        lut[2, 1] = IMPROVEMENT
        lut[2, 3] = IMPROVEMENT

        # Row Cropland (3):
        lut[3, 1] = IMPROVEMENT

        # Row Wetlands (4):

        # Row Artificial (5):
        lut[5, 1] = IMPROVEMENT
        lut[5, 2] = IMPROVEMENT
        lut[5, 3] = IMPROVEMENT
        lut[5, 4] = IMPROVEMENT
        lut[5, 6] = IMPROVEMENT

        # Row Other (6):
        lut[6, 1] = IMPROVEMENT
        lut[6, 2] = IMPROVEMENT
        lut[6, 3] = IMPROVEMENT
        lut[6, 4] = IMPROVEMENT
    elif period == 'Reporting':
        # Row Forest (1):

        # Row Grassland (2):
        lut[2, 1] = IMPROVEMENT
        lut[2, 3] = IMPROVEMENT
        lut[2, 6] = STABLE

        # Row Cropland (3):
        lut[3, 1] = IMPROVEMENT

        # Row Wetlands (4):

        # Row Artificial (5):
        lut[5, 1] = IMPROVEMENT
        lut[5, 2] = IMPROVEMENT
        lut[5, 3] = IMPROVEMENT
        lut[5, 4] = IMPROVEMENT
        lut[5, 6] = IMPROVEMENT

        # Row Other (6):
        lut[6, 1] = IMPROVEMENT
        lut[6, 2] = STABLE
        lut[6, 3] = IMPROVEMENT
        lut[6, 4] = IMPROVEMENT

    return lut


def compute_status(cls0, cls1, nodata=NODATA_VAL, ignore_classes=IGNORE_CLASSES, period='Baseline'):
    """
    cls0, cls1: arrays of land cover classes (expected 1..7), with nodata=255 after mask.
    returns: status array (uint8): 255 NoData, 0 Stable, 1 Improvement, 2 Degradation
    """
    cls0 = cls0.astype(np.int16, copy=False)
    cls1 = cls1.astype(np.int16, copy=False)

    valid = (cls0 != nodata) & (cls1 != nodata)
    for c in ignore_classes:
        valid &= (cls0 != c) & (cls1 != c)

    out = np.full(cls0.shape, nodata, dtype=np.uint8)

    lut = build_transition_lut(nodata=nodata, period=period)

    # apply LUT only to valid pixels
    out[valid] = lut[cls0[valid], cls1[valid]]
    return out


def _safe_nodata_for_dtype(nodata, dtype):
    dt = np.dtype(dtype)
    if np.issubdtype(dt, np.integer):
        info = np.iinfo(dt)
        n = int(nodata)
        if n < info.min or n > info.max:
            return 0 if (0 >= info.min and 0 <= info.max) else max(min(n, info.max), info.min)
        return n
    return float(nodata)


def _reproject_to_equal_area_for_area(ds, arr, ea_epsg=6933, nodata=NODATA_VAL):
    """
    Reproject categorical raster to equal-area CRS for area stats.
    """
    src_arr = arr.astype(np.int16, copy=False)
    src_nodata = _safe_nodata_for_dtype(nodata, src_arr.dtype)

    dst_crs = rasterio.crs.CRS.from_epsg(ea_epsg)
    transform, width, height = calculate_default_transform(
        ds.crs, dst_crs, ds.width, ds.height, *ds.bounds
    )

    dst = np.full((height, width), src_nodata, dtype=np.int16)

    reproject(
        source=src_arr,
        destination=dst,
        src_transform=ds.transform,
        src_crs=ds.crs,
        dst_transform=transform,
        dst_crs=dst_crs,
        resampling=Resampling.nearest,
        src_nodata=src_nodata,
        dst_nodata=src_nodata,
    )

    pixel_area_km2 = abs(transform.a * transform.e) / 1e6
    return dst, transform, dst_crs, pixel_area_km2


def area_stats_km2(ds_ref, arr_uint8, nodata=NODATA_VAL):
    """
    Compute area by class code in km2.
    - If projected CRS (meters), use constant pixel area.
    - Otherwise reproject to equal-area (EPSG:6933) for stable pixel area.
    """
    if ds_ref.crs is not None and ds_ref.crs.is_projected:
        px_km2 = abs(ds_ref.transform.a * ds_ref.transform.e) / 1e6
        vals, counts = np.unique(arr_uint8, return_counts=True)
        areas = counts * px_km2
        return vals, areas
    else:
        ea_arr, _, _, px_km2 = _reproject_to_equal_area_for_area(ds_ref, arr_uint8, ea_epsg=6933, nodata=nodata)
        vals, counts = np.unique(ea_arr, return_counts=True)
        areas = counts * px_km2
        return vals, areas


# =========================
# Main
# =========================
os.makedirs(os.path.dirname(OUT_TIF), exist_ok=True)

with rasterio.open(TIF_T0) as ds0, rasterio.open(TIF_T1) as ds1:
    # AOI -> raster CRS
    aoi_geoms = load_aoi_geoms(AOI_SHP, ds0.crs)

    # Mask/crop to AOI. AOI outside becomes nodata=255.
    arr0, out_transform = mask(ds0, aoi_geoms, crop=True, nodata=NODATA_VAL, filled=True)
    arr1, _             = mask(ds1, aoi_geoms, crop=True, nodata=NODATA_VAL, filled=True)

    cls0 = arr0[0]
    cls1 = arr1[0]

    # If the two masked rasters still differ in grid (rare but possible), align cls1 -> cls0
    # We need temporary dataset objects for reproject; simplest is to re-open cropped windows is complex.
    # In practice, your exports are aligned; still, we can do a lightweight check and warn.
    if cls0.shape != cls1.shape:
        raise ValueError(f"Masked rasters have different shapes: {cls0.shape} vs {cls1.shape}. "
                         "This usually means the two inputs are not on the same grid. "
                         "Export them with identical region/projection first.")

    # Compute status
    status = compute_status(cls0, cls1, nodata=NODATA_VAL, ignore_classes=IGNORE_CLASSES, period=PERIOD)

    # Optional quick sanity check
    u, c = np.unique(status, return_counts=True)
    print("Unique status codes:", dict(zip(u.tolist(), c.tolist())))

    # Write status GeoTIFF
    profile = ds0.profile.copy()
    profile.update(
        height=status.shape[0],
        width=status.shape[1],
        transform=out_transform,
        count=1,
        dtype=rasterio.uint8,
        nodata=NODATA_VAL,
        compress="lzw",
        tiled=True,
    )

    with rasterio.open(OUT_TIF, "w", **profile) as dst:
        dst.write(status.astype(np.uint8), 1)
        dst.write_colormap(1, STATUS_COLORMAP)

print("Saved:", OUT_TIF)

# Area stats
with rasterio.open(OUT_TIF) as ds_ref:
    status_arr = ds_ref.read(1)
    vals, areas = area_stats_km2(ds_ref, status_arr, nodata=NODATA_VAL)

# Remove NoData from stats (optional; keep if you want AOI-window outside-polygon area)
mask_valid = vals != NODATA_VAL
vals = vals[mask_valid]
areas = areas[mask_valid]

df = pd.DataFrame({
    "status_code": vals.astype(int),
    "label": [STATUS_LABELS.get(int(v), "Unknown") for v in vals],
    "area_km2": areas
}).sort_values("status_code")

df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
print("Saved:", OUT_CSV)
print(df)

Unique status codes: {0: 860561, 1: 1578, 2: 4610, 255: 1448452}
Saved: ../Result/Land_Cover_Degradation/status_2018_2019_deg_stable_imp.tif
Saved: ../Result/Land_Cover_Degradation/status_2018_2019_area_km2_stats.csv
   status_code        label      area_km2
0            0       Stable  70489.280956
1            1  Improvement    128.163462
2            2  Degradation    369.540576
